# TranAD+CL 在 SMAP 上复现实验

按顺序运行下面代码单元：
1. 环境与参数
2. 依赖与数据检查
3. 导入与兼容补丁
4. Baseline 训练评估
5. CL 冻结主干
6. alpha 自动搜索与最优结果

In [ ]:
from pathlib import Path
import os
import os
import sys
from pathlib import Path

TRANAD_ROOT = Path(os.environ.get("TRANAD_ROOT", "../TranAD")).resolve()
assert TRANAD_ROOT.is_dir(), f"未找到 TranAD 目录: {TRANAD_ROOT}"

os.chdir(TRANAD_ROOT)
if str(TRANAD_ROOT) not in sys.path:
    sys.path.insert(0, str(TRANAD_ROOT))

# 与命令行等价
sys.argv = [
    "tranad_smap_reproduce.ipynb",
    "--dataset", "SMAP",
    "--model", "TranAD",
    "--retrain",
]

print("cwd:", os.getcwd())
print("argv:", sys.argv)
print("python:", sys.executable)

In [ ]:
# Matplotlib 字体与警告抑制（放在所有绘图之前）
import logging
import warnings
import matplotlib as mpl
from matplotlib import font_manager as fm

# 1) 关闭 font_manager 噪声日志
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# 2) 屏蔽常见字体警告
warnings.filterwarnings("ignore", message="findfont: Generic family")
warnings.filterwarnings("ignore", message=r"Glyph .* missing from font\(s\)")

# 3) 自动选择本机可用字体，避免回退到 Times
try:
    available = {f.name for f in fm.fontManager.ttflist}
except Exception:
    available = set()

preferred_zh = ["Microsoft YaHei", "SimHei", "SimSun", "Noto Sans CJK SC", "PingFang SC"]
zh_fonts = [f for f in preferred_zh if f in available]
if not zh_fonts:
    zh_fonts = ["Microsoft YaHei", "SimHei", "SimSun"]

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = zh_fonts + ["Arial", "DejaVu Sans"]
# 关键：给 serif 提供可用兜底，避免去找 Times
mpl.rcParams["font.serif"] = ["DejaVu Serif", "Times New Roman", "SimSun"]
mpl.rcParams["axes.unicode_minus"] = False

print("Matplotlib 字体配置完成：", mpl.rcParams["font.sans-serif"][:3], "...")
print("已抑制 findfont / glyph 缺失警告。")

In [ ]:
# Install dependencies explicitly using docs/reproduction.md before running.
import subprocess
import torchdata.datapipes
import pydantic
# 数据检查
data_root = TRANAD_ROOT / "data" / "SMAP_MSL"
assert (data_root / "train").is_dir(), "缺少 data/SMAP_MSL/train"
assert (data_root / "test").is_dir(), "缺少 data/SMAP_MSL/test"
assert (data_root / "labeled_anomalies.csv").is_file(), "缺少 labeled_anomalies.csv"

# 预处理（只在缺失时执行）
processed_dir = TRANAD_ROOT / "processed" / "SMAP"
need = [processed_dir / "P-1_train.npy", processed_dir / "P-1_test.npy", processed_dir / "P-1_labels.npy"]
if all(p.exists() for p in need):
    print("processed/SMAP 已就绪，跳过 preprocess")
else:
    print("开始 preprocess.py SMAP ...")
    subprocess.check_call([sys.executable, "preprocess.py", "SMAP"])
    print("SMAP 预处理完成")

In [ ]:
import importlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from time import time
from pprint import pprint
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset

import src.dlutils as dlutils

# PyTorch 2.x is_causal 兼容补丁
_enc_orig = dlutils.TransformerEncoderLayer.forward
def _enc_patched(self, src, src_mask=None, src_key_padding_mask=None, **kwargs):
    kwargs.pop("is_causal", None)
    return _enc_orig(self, src, src_mask, src_key_padding_mask)
dlutils.TransformerEncoderLayer.forward = _enc_patched

_dec_orig = dlutils.TransformerDecoderLayer.forward
def _dec_patched(self, tgt, memory, tgt_mask=None, memory_mask=None,
                 tgt_key_padding_mask=None, memory_key_padding_mask=None, **kwargs):
    kwargs.pop("is_causal", None)
    return _dec_orig(self, tgt, memory, tgt_mask, memory_mask,
                     tgt_key_padding_mask, memory_key_padding_mask)
dlutils.TransformerDecoderLayer.forward = _dec_patched

import src.constants as constants
importlib.reload(constants)
from src.constants import *
from src.pot import pot_eval
from src.diagnosis import hit_att, ndcg
from src.utils import plot_accuracies
from main import load_dataset, convert_to_windows, backprop, save_model, load_model

constants.preds.clear()
print("patch 已应用")
print("dataset:", args.dataset, "model:", args.model, "retrain:", args.retrain, "test:", args.test)

In [ ]:
train_loader, test_loader, labels = load_dataset(args.dataset)
model, optimizer, scheduler, epoch, accuracy_list = load_model(args.model, labels.shape[1])

trainD, testD = next(iter(train_loader)), next(iter(test_loader))
trainO, testO = trainD, testD
trainD, testD = convert_to_windows(trainD, model), convert_to_windows(testD, model)

if not args.test:
    print(f"Training {args.model} on {args.dataset}")
    num_epochs = 5
    start = time()
    for e in tqdm(list(range(epoch + 1, epoch + num_epochs + 1)), desc="epochs"):
        lossT, lr = backprop(e, model, trainD, trainO, optimizer, scheduler)
        accuracy_list.append((lossT, lr))
    print("Training time: {:10.4f} s".format(time() - start))
    save_model(model, optimizer, scheduler, e, accuracy_list)
    plot_accuracies(accuracy_list, f"{args.model}_{args.dataset}")

model.eval()
print(f"Testing {args.model} on {args.dataset}")
with torch.no_grad():
    loss, _ = backprop(0, model, testD, testO, optimizer, scheduler, training=False)

with torch.no_grad():
    lossT, _ = backprop(0, model, trainD, trainO, optimizer, scheduler, training=False)

lossTfinal, lossFinal = np.mean(lossT, axis=1), np.mean(loss, axis=1)
labelsFinal = (np.sum(labels, axis=1) >= 1).astype(int)
result, _ = pot_eval(lossTfinal, lossFinal, labelsFinal)
result.update(hit_att(loss, labels))
result.update(ndcg(loss, labels))
baseline_result = dict(result)

print("\n--- Baseline TranAD ---")
pprint(result)

In [ ]:
need_vars = ["model", "trainD", "testD", "lossTfinal", "lossFinal", "labels", "result", "pot_eval"]
miss = [v for v in need_vars if v not in globals()]
if miss:
    raise RuntimeError(f"缺少变量 {miss}，请先运行上一格。")

CL_EPOCHS = 5
CL_BATCH = 256
CL_LR = 8e-4
TEMPERATURE = 0.2

def aug_jitter(x, sigma=0.01):
    return x + torch.randn_like(x) * sigma

def aug_scaling(x, sigma=0.05):
    b, _, d = x.shape
    s = torch.normal(mean=torch.ones(b, 1, d, device=x.device, dtype=x.dtype), std=sigma)
    return x * s

def aug_feature_mask(x, ratio=0.05):
    if ratio <= 0:
        return x
    b, _, d = x.shape
    m = max(1, int(d * ratio))
    y = x.clone()
    for i in range(b):
        idx = torch.randperm(d, device=x.device)[:m]
        y[i, :, idx] = 0.0
    return y

def augment_ts_cl(x):
    return aug_feature_mask(aug_scaling(aug_jitter(x, 0.01), 0.05), 0.05)

def info_nce(z1, z2, temperature=0.2):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    logits = (z1 @ z2.t()) / temperature
    labels = torch.arange(z1.size(0), device=z1.device, dtype=torch.long)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels))

device_ = next(model.parameters()).device
dtype_ = next(model.parameters()).dtype
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

with torch.no_grad():
    xb = trainD[:min(128, len(trainD))].to(device=device_, dtype=dtype_)
    src = xb.permute(1, 0, 2)
    tgt = src[-1:, :, :]
    c = torch.zeros_like(src)
    _, mem = model.encode(src, c, tgt)
    rep_dim = mem.mean(dim=0).shape[1]

proj_head = nn.Sequential(
    nn.Linear(rep_dim, rep_dim),
    nn.ReLU(inplace=True),
    nn.Linear(rep_dim, 128),
).to(device=device_, dtype=dtype_)

optimizer_head = torch.optim.AdamW(proj_head.parameters(), lr=CL_LR, weight_decay=1e-5)
loader = DataLoader(TensorDataset(trainD, trainD), batch_size=CL_BATCH, shuffle=True)

for ep in range(1, CL_EPOCHS + 1):
    proj_head.train()
    ep_loss, n = 0.0, 0
    for xb, _ in loader:
        xb = xb.to(device=device_, dtype=dtype_)
        x1, x2 = augment_ts_cl(xb), augment_ts_cl(xb)

        with torch.no_grad():
            s1 = x1.permute(1, 0, 2)
            t1 = s1[-1:, :, :]
            c1 = torch.zeros_like(s1)
            _, m1 = model.encode(s1, c1, t1)
            r1 = m1.mean(dim=0)

            s2 = x2.permute(1, 0, 2)
            t2 = s2[-1:, :, :]
            c2 = torch.zeros_like(s2)
            _, m2 = model.encode(s2, c2, t2)
            r2 = m2.mean(dim=0)

        z1, z2 = proj_head(r1), proj_head(r2)
        loss_cl = info_nce(z1, z2, TEMPERATURE)

        optimizer_head.zero_grad(set_to_none=True)
        loss_cl.backward()
        optimizer_head.step()

        bs = xb.size(0)
        ep_loss += float(loss_cl.item()) * bs
        n += bs

    print(f"[CL-frozen] epoch={ep:02d} infoNCE={ep_loss/max(n,1):.6f}")

def extract_cl_score(x_windows):
    zs = []
    dl = DataLoader(TensorDataset(x_windows, x_windows), batch_size=CL_BATCH, shuffle=False)
    proj_head.eval()
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(device=device_, dtype=dtype_)
            s = xb.permute(1, 0, 2)
            t = s[-1:, :, :]
            c = torch.zeros_like(s)
            _, mem = model.encode(s, c, t)
            r = mem.mean(dim=0)
            z = F.normalize(proj_head(r), dim=1)
            zs.append(z.cpu().numpy())
    return np.concatenate(zs, axis=0)

z_train = extract_cl_score(trainD)
z_test = extract_cl_score(testD)
center = z_train.mean(axis=0, keepdims=True)
center = center / (np.linalg.norm(center, axis=1, keepdims=True) + 1e-8)

score_train_cl = 1.0 - np.sum(z_train * center, axis=1)
score_test_cl = 1.0 - np.sum(z_test * center, axis=1)

print("CL score 已生成: score_train_cl / score_test_cl")

In [ ]:
need = ["lossTfinal", "lossFinal", "labels", "result", "pot_eval", "score_train_cl", "score_test_cl"]
miss = [v for v in need if v not in globals()]
if miss:
    raise RuntimeError(f"缺少变量 {miss}，请先运行前面单元。")

base_tr = np.asarray(lossTfinal, dtype=np.float64).ravel()
base_te = np.asarray(lossFinal, dtype=np.float64).ravel()
cl_tr = np.asarray(score_train_cl, dtype=np.float64).ravel()
cl_te = np.asarray(score_test_cl, dtype=np.float64).ravel()
labels_final = (np.sum(labels, axis=1) >= 1).astype(int)

# 先把 CL 分数对齐到 baseline 尺度
mu_b, std_b = base_tr.mean(), base_tr.std() + 1e-12
mu_c, std_c = cl_tr.mean(), cl_tr.std() + 1e-12
cl_tr_align = (cl_tr - mu_c) / std_c * std_b + mu_b
cl_te_align = (cl_te - mu_c) / std_c * std_b + mu_b

alpha_grid = np.round(np.arange(0.00, 0.41, 0.02), 2)
scan = []
for a in alpha_grid:
    fuse_tr = (1.0 - a) * base_tr + a * cl_tr_align
    fuse_te = (1.0 - a) * base_te + a * cl_te_align
    res, pred = pot_eval(fuse_tr, fuse_te, labels_final)
    scan.append({
        "alpha": float(a),
        "f1": float(res.get("f1", np.nan)),
        "precision": float(res.get("precision", np.nan)),
        "recall": float(res.get("recall", np.nan)),
        "ROC/AUC": float(res.get("ROC/AUC", np.nan)),
        "threshold": float(res.get("threshold", np.nan)),
        "TP": float(res.get("TP", np.nan)),
        "TN": float(res.get("TN", np.nan)),
        "FP": float(res.get("FP", np.nan)),
        "FN": float(res.get("FN", np.nan)),
        "res": res,
        "pred": pred,
        "fuse_tr": fuse_tr,
        "fuse_te": fuse_te,
    })

scan_df = pd.DataFrame([{k: v for k, v in row.items() if k not in ["res", "pred", "fuse_tr", "fuse_te"]} for row in scan])
scan_df = scan_df.sort_values(["f1", "precision", "recall"], ascending=[False, False, False]).reset_index(drop=True)
best_alpha = float(scan_df.loc[0, "alpha"])

best_row = next(r for r in scan if abs(r["alpha"] - best_alpha) < 1e-12)
result_cl_best = best_row["res"]
pred_cl_best = best_row["pred"]
fuse_tr_best = best_row["fuse_tr"]
fuse_te_best = best_row["fuse_te"]

print(f"best alpha = {best_alpha:.2f}")
print("\n--- TranAD + CL(best alpha) ---")
for k, v in result_cl_best.items():
    if isinstance(v, (float, np.floating)):
        print(f"  {k}: {float(v):.6f}")
    else:
        print(f"  {k}: {v}")

keys = ["f1", "precision", "recall", "ROC/AUC", "threshold", "TP", "TN", "FP", "FN"]
cmp_df = pd.DataFrame({
    "metric": keys,
    "baseline_tranad": [baseline_result.get(k, np.nan) for k in keys],
    "tranad_cl_best_alpha": [result_cl_best.get(k, np.nan) for k in keys],
})
display(cmp_df)

display(scan_df.head(10))

best_alpha_artifacts = {
    "alpha_best": best_alpha,
    "scan_df": scan_df,
    "result": result_cl_best,
    "pred": pred_cl_best,
    "fuse_tr": fuse_tr_best,
    "fuse_te": fuse_te_best,
}
print("已生成 best_alpha_artifacts")

## 6) 可视化增强（SMAP）

下面几格参考 WADI 笔记本，补充 alpha 搜索与最终结果的图形化分析。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

need = ["scan_df", "baseline_result", "best_alpha_artifacts"]
miss = [v for v in need if v not in globals()]
if miss:
    raise RuntimeError(f"缺少变量 {miss}，请先运行 alpha 搜索单元。")

alpha_best = float(best_alpha_artifacts["alpha_best"])

# 1) alpha 与指标曲线
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(scan_df["alpha"], scan_df["f1"], marker="o", label="F1")
ax.plot(scan_df["alpha"], scan_df["precision"], marker="s", label="Precision", alpha=0.9)
ax.plot(scan_df["alpha"], scan_df["recall"], marker="^", label="Recall", alpha=0.9)
ax.axvline(alpha_best, color="C3", linestyle="--", alpha=0.8, label=f"best alpha={alpha_best:.2f}")
ax.set_xlabel("alpha")
ax.set_ylabel("metric")
ax.set_title("Frozen CL fusion: metric vs alpha (SMAP)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# 2) baseline vs best-alpha 条形图
keys = ["f1", "precision", "recall", "ROC/AUC"]
b = [float(baseline_result.get(k, np.nan)) for k in keys]
c = [float(best_alpha_artifacts["result"].get(k, np.nan)) for k in keys]

x = np.arange(len(keys))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - w / 2, b, width=w, label="TranAD")
ax.bar(x + w / 2, c, width=w, label=f"TranAD+CL (alpha={alpha_best:.2f})")
ax.set_xticks(x)
ax.set_xticklabels(keys)
ax.set_ylim(0, 1.05)
ax.grid(True, axis="y", alpha=0.3)
ax.set_title("Baseline vs TranAD+CL (SMAP)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)

need = ["lossTfinal", "lossFinal", "labels", "pot_eval", "best_alpha_artifacts"]
miss = [v for v in need if v not in globals()]
if miss:
    raise RuntimeError(f"缺少变量 {miss}，请先运行前面单元。")

base_tr = np.asarray(lossTfinal, dtype=np.float64).ravel()
base_te = np.asarray(lossFinal, dtype=np.float64).ravel()
labels_final = (np.sum(labels, axis=1) >= 1).astype(int)

fuse_tr = np.asarray(best_alpha_artifacts["fuse_tr"], dtype=np.float64).ravel()
fuse_te = np.asarray(best_alpha_artifacts["fuse_te"], dtype=np.float64).ravel()
alpha_best = float(best_alpha_artifacts["alpha_best"])

res_base, pred_base = pot_eval(base_tr, base_te, labels_final)
res_cl, pred_cl = pot_eval(fuse_tr, fuse_te, labels_final)

# 1) 混淆矩阵对比
cm_base = confusion_matrix(labels_final, pred_base, labels=[0, 1])
cm_cl = confusion_matrix(labels_final, pred_cl, labels=[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cm, title in [(axes[0], cm_base, "TranAD"), (axes[1], cm_cl, "TranAD+CL")]:
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["normal", "abnormal"])
    ax.set_yticklabels(["normal", "abnormal"])
    ax.set_xlabel("Pred"); ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
fig.suptitle("Confusion Matrix Comparison")
plt.tight_layout()
plt.show()

# 2) ROC / PR 对比
fpr_b, tpr_b, _ = roc_curve(labels_final, base_te)
fpr_c, tpr_c, _ = roc_curve(labels_final, fuse_te)
auc_b, auc_c = auc(fpr_b, tpr_b), auc(fpr_c, tpr_c)

fig, ax = plt.subplots(figsize=(6.2, 5))
ax.plot(fpr_b, tpr_b, lw=2, label=f"TranAD (AUC={auc_b:.3f})")
ax.plot(fpr_c, tpr_c, lw=2, label=f"TranAD+CL (AUC={auc_c:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC Curve")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

prec_b, rec_b, _ = precision_recall_curve(labels_final, base_te)
prec_c, rec_c, _ = precision_recall_curve(labels_final, fuse_te)
ap_b, ap_c = average_precision_score(labels_final, base_te), average_precision_score(labels_final, fuse_te)

fig, ax = plt.subplots(figsize=(6.2, 5))
ax.plot(rec_b, prec_b, lw=2, label=f"TranAD (AP={ap_b:.3f})")
ax.plot(rec_c, prec_c, lw=2, label=f"TranAD+CL (AP={ap_c:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# 3) 分数分布与时序对比
mask_n = labels_final == 0
mask_a = labels_final == 1
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, scores, title in [(axes[0], base_te, "TranAD"), (axes[1], fuse_te, f"TranAD+CL (alpha={alpha_best:.2f})")]:
    ax.hist(scores[mask_n], bins=60, alpha=0.55, density=True, label="normal")
    ax.hist(scores[mask_a], bins=60, alpha=0.55, density=True, label="abnormal")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend()
fig.suptitle("Score Distribution")
plt.tight_layout(); plt.show()

n_show = min(2000, len(base_te))
t = np.arange(n_show)
y_true = labels_final[:n_show]
score_b = base_te[:n_show]
score_c = fuse_te[:n_show]

fig, ax = plt.subplots(figsize=(12, 4.2))
ax.plot(t, score_b, lw=1.2, label="TranAD score")
ax.plot(t, score_c, lw=1.2, label="TranAD+CL score")
ax.fill_between(t, min(score_b.min(), score_c.min()), max(score_b.max(), score_c.max()), where=(y_true > 0), color="red", alpha=0.10, label="true anomaly")
ax.set_title(f"Score Timeline (first {n_show}), alpha_best={alpha_best:.2f}")
ax.set_xlabel("time index")
ax.set_ylabel("score")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

print("可视化完成：混淆矩阵、ROC/PR、分布图、时序图。")
print("TranAD:", {k: res_base.get(k) for k in ["f1", "precision", "recall", "ROC/AUC"]})
print("TranAD+CL:", {k: res_cl.get(k) for k in ["f1", "precision", "recall", "ROC/AUC"]})